In [0]:
# CELL 1 — Sources and experiment configuration
demand_path = "development_042_silver_sandbox.demand_forecast.demand_forecast_results"
ai_attr_silver_path = "development_042_silver_sandbox.demand_forecast.game_attr_ai_silver"

FORECAST_MONTHS = 6  # Set to 12 to forecast the full available horizon.
TEST_FRACTION = 0.20
CAP_QUANTILE = 0.99
RANDOM_SEED = 42

MLFLOW_EXPERIMENT_PATH = None  # Default: /Users/<current_user>/AI_feature_extraction_forecasts
MLFLOW_RUN_NAME = None         # Default: xgb_ai_attributes_h<FORECAST_MONTHS>_tuned_v4
MLFLOW_LOG_DATA_SNAPSHOT = True

XGB_PARAMS = dict(
    objective="reg:squarederror", n_estimators=300, learning_rate=0.05,
    max_depth=3, min_child_weight=10, subsample=0.9, colsample_bytree=0.8,
    reg_lambda=10.0, tree_method="hist", random_state=RANDOM_SEED, n_jobs=4,
)
if not isinstance(FORECAST_MONTHS, int) or not 1 <= FORECAST_MONTHS <= 12:
    raise ValueError("FORECAST_MONTHS must be an integer between 1 and 12.")
if not 0 < TEST_FRACTION < 1:
    raise ValueError("TEST_FRACTION must be between 0 and 1.")
if not 0 < CAP_QUANTILE <= 1:
    raise ValueError("CAP_QUANTILE must be between 0 and 1.")

# Tuning uses earlier TRAINING games only; final TEST games remain untouched.
# Trial 0 always evaluates XGB_PARAMS. Remaining candidates are random samples.
TUNE_HYPERPARAMETERS = True
TUNING_TRIALS = 20  # Maximum candidates per target, including the configured default.
TUNING_FOLDS = 3
TUNING_MIN_FOLDS = 2
TUNING_INITIAL_TRAIN_FRACTION = 0.50
TUNING_MIN_TRAIN_GROUPS = 20
TUNING_MIN_VALIDATION_GROUPS = 5
TUNING_METRIC = "RMSE"  # "RMSE" or "MAE"; validation gives each game equal weight.
TUNING_SEARCH_SPACE = {
    "n_estimators": [150, 300, 600],
    "learning_rate": [0.02, 0.05, 0.10],
    "max_depth": [2, 3, 4, 5],
    "min_child_weight": [3, 10, 20],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "reg_lambda": [1.0, 5.0, 10.0, 25.0],
    "reg_alpha": [0.0, 0.5, 2.0],
    "gamma": [0.0, 1.0, 5.0],
}

for name in ["TUNING_TRIALS", "TUNING_FOLDS", "TUNING_MIN_FOLDS",
             "TUNING_MIN_TRAIN_GROUPS", "TUNING_MIN_VALIDATION_GROUPS"]:
    if type(globals()[name]) is not int or globals()[name] < 1:
        raise ValueError(f"{name} must be a positive integer.")
if not 2 <= TUNING_MIN_FOLDS <= TUNING_FOLDS:
    raise ValueError("Require 2 <= TUNING_MIN_FOLDS <= TUNING_FOLDS.")
if not 0 < TUNING_INITIAL_TRAIN_FRACTION < 1:
    raise ValueError("TUNING_INITIAL_TRAIN_FRACTION must be between 0 and 1.")
if TUNING_METRIC not in {"RMSE", "MAE"}:
    raise ValueError("TUNING_METRIC must be RMSE or MAE.")
if not TUNING_SEARCH_SPACE or any(not isinstance(v, (list, tuple)) or not v
                                 for v in TUNING_SEARCH_SPACE.values()):
    raise ValueError("TUNING_SEARCH_SPACE must contain nonempty lists of parameter values.")
reserved = {"monotone_constraints", "early_stopping_rounds", "callbacks"}
if reserved & (set(XGB_PARAMS) | set(TUNING_SEARCH_SPACE)):
    raise ValueError("Constraints are set automatically; tune n_estimators instead of early stopping.")


In [0]:
# CELL 2 — Load game attributes and observed demand targets
df_demand_forecast = spark.table(demand_path)
ai_attr = spark.table(ai_attr_silver_path)
SOURCE_TABLES = {"demand": demand_path, "attributes": ai_attr_silver_path}

display(df_demand_forecast)
display(ai_attr)


In [0]:
# CELL 3 — Clean attributes and prepare observed targets. Learned transforms are in cell 4.
from pyspark.sql import functions as F, types as T

# ============================================================
# 1. Helpers
# ============================================================
NULL_NAMES = ["", "none", "null", "nan", "na", "n/a"]


def clean_name(column):
    value = F.regexp_replace(
        F.lower(F.trim(column.cast("string"))),
        r"\s+",
        " ",
    )
    return F.when(~value.isin(NULL_NAMES), value)


def numeric_value(df, column_name, preferred_fields):
    """Extract a numeric scalar from a scalar or nested AI field."""
    dtype = df.schema[column_name].dataType

    if isinstance(dtype, T.StructType):
        fields = dtype.fieldNames()
        selected = next(
            (name for name in preferred_fields if name in fields),
            None,
        )

        # Accommodate alternate extraction schemas.
        if selected is None:
            numeric_fields = [
                field.name
                for field in dtype.fields
                if isinstance(field.dataType, T.NumericType)
            ]
            if len(numeric_fields) == 1:
                selected = numeric_fields[0]

        if selected is None:
            raise ValueError(
                f"Cannot identify the numeric field in {column_name}: "
                f"{fields}. Add its field name to preferred_fields."
            )

        value = F.col(column_name).getField(selected)
    else:
        value = F.col(column_name)

    # try_cast tolerates malformed extraction values.
    value = value.try_cast("double")
    return F.when(
        value.isNotNull()
        & ~F.isnan(value)
        & (F.abs(value) != F.lit(float("inf"))),
        value,
    )


def require_unique(df, keys, description):
    duplicates = (
        df.groupBy(*keys)
        .count()
        .where(F.col("count") > 1)
    )
    if duplicates.limit(1).count():
        display(duplicates.limit(20))
        raise ValueError(f"{description}: duplicated keys {keys}.")


# ============================================================
# 2. Clean AI attributes: one row per game
# ============================================================
theme_columns = sorted(
    name for name in ai_attr.columns
    if name.startswith("theme_") and name.endswith("_score")
)

mechanic_columns = sorted(
    name for name in ai_attr.columns
    if name.startswith("mechanic_") and name.endswith("_present")
)

# Use the curated key. Keep original names as metadata.
attrs = (
    ai_attr
    .withColumn("game_key", clean_name(F.col("join_game_name_key")))
    .where(F.col("game_key").isNotNull())
    .dropDuplicates()
)

from pyspark.sql.window import Window

game_row_window = Window.partitionBy("game_key").orderBy(
    F.when(
        F.lower(F.trim(F.col("parse_status"))) == "valid_json", 1
    ).otherwise(0).desc(),
    F.col("saved_at").desc_nulls_last(),
    F.col("run_id").desc_nulls_last(),
    F.col("game_name").asc_nulls_last(),
)

attrs = (
    attrs
    .withColumn("_row_number", F.row_number().over(game_row_window))
    .where(F.col("_row_number") == 1)
    .drop("_row_number")
)
attrs = (
    attrs
    .withColumn(
        "reels",
        numeric_value(
            attrs, "number_of_reels",
            ["value", "count", "number_of_reels", "number"],
        ),
    )
    .withColumn(
        "paylines_or_ways",
        numeric_value(
            attrs, "paylines_or_ways_count",
            ["value", "count", "paylines_or_ways_count", "number"],
        ),
    )
    .withColumn(
        "persistence_score",
        numeric_value(
            attrs, "persistence",
            ["score", "value", "persistence_score"],
        ),
    )
    .withColumn(
        "perceived_persistence_score",
        numeric_value(
            attrs, "perceived_persistence",
            ["score", "value", "perceived_persistence_score"],
        ),
    )
)

# Invalid counts become missing. Learn upper caps from training later.
for name in ["reels", "paylines_or_ways"]:
    attrs = attrs.withColumn(
        name,
        F.when(F.col(name) > 0, F.col(name)),
    )

# Expected score ranges; preserve unknown persistence as null.
attrs = (
    attrs
    .withColumn(
        "persistence_score",
        F.when(
            F.col("persistence_score").between(0, 1),
            F.col("persistence_score"),
        ),
    )
    .withColumn(
        "perceived_persistence_score",
        F.when(
            F.col("perceived_persistence_score").between(0, 5),
            F.col("perceived_persistence_score"),
        ),
    )
)

# On successfully extracted games, null category entries mean absent.
# Failed/unknown extractions retain null rather than becoming all-zero.
extraction_ok = (
    F.lower(F.trim(F.col("parse_status"))) == "valid_json"
)

for name in theme_columns:
    value = F.col(name).try_cast("double")
    attrs = attrs.withColumn(
        name,
        F.when(value.between(0, 5), value)
        .when(value.isNull() & extraction_ok, F.lit(0.0)),
    )

for name in mechanic_columns:
    value = F.col(name).try_cast("double")
    attrs = attrs.withColumn(
        name,
        F.when(value.isin(0.0, 1.0), value)
        .when(value.isNull() & extraction_ok, F.lit(0.0)),
    )

# A self-parent entry is not a separate parent game.
parent_key = clean_name(F.col("parent_game"))
original_name_key = clean_name(
    F.regexp_replace(
        F.lower(F.col("game_name")),
        r"\s+overview(?:-.*)?$",
        "",
    )
)

attrs = attrs.withColumn(
    "has_parent_game",
    F.when(parent_key.isNull(), F.lit(None).cast("double"))
    .when(
        (parent_key == F.col("game_key"))
        | (parent_key == original_name_key),
        F.lit(0.0),
    )
    .otherwise(F.lit(1.0)),
)

categorical_features = ["game_category", "game_classification"]

for name in categorical_features:
    attrs = attrs.withColumn(name, clean_name(F.col(name)))

numeric_features = [
    "reels",
    "paylines_or_ways",
    "persistence_score",
    "perceived_persistence_score",
    "has_parent_game",
] + theme_columns + mechanic_columns

metadata_columns = [
    "game_key",
    "game_name",
    "game_release_date",
    "to_sales_name_match",
    "to_rental_name_match",
]

ai_attr_clean_df = attrs.select(
    *metadata_columns,
    *numeric_features,
    *categorical_features,
)


# ============================================================
# 3. Extract targets, excluding similarity forecast values
# ============================================================
revenue = F.lower(F.trim(F.col("revenue_type")))

targets = (
    df_demand_forecast
    .withColumn("game_key", clean_name(F.col("target_game")))
    .withColumn(
        "target_type",
        F.when(revenue.isin("sale", "sales"), "sales")
        .when(revenue.isin("lease", "rental", "rent"), "rental"),
    )
    .withColumn("month_index", F.col("month_index").cast("int"))
    .withColumn(
        "target_value",
        F.when(
            F.col("target_type") == "sales",
            F.col("actual_cumulative_sales").cast("double"),
        ).otherwise(F.col("actual_monthly_lease").cast("double")),
    )
    .withColumn(
        "source_start_date",
        F.trunc(F.to_date("curve_start_month"), "month"),
    )
    .withColumn(
        "target_is_imputed",
        F.col("actual_is_imputed").cast("boolean"),
    )
    .where(
        F.col("game_key").isNotNull()
        & F.col("target_type").isNotNull()
        & F.col("month_index").between(1, 12)
    )
)

# Actuals repeat across similarity experiments.
# Check for disagreements before collapsing those repetitions.
target_keys = ["game_key", "target_type", "month_index"]

conflicts = (
    targets.groupBy(*target_keys)
    .agg(
        F.countDistinct("target_value").alias("different_actuals"),
        F.countDistinct("source_start_date").alias("different_starts"),
        F.countDistinct("target_is_imputed").alias("different_flags"),
    )
    .where(
        (F.col("different_actuals") > 1)
        | (F.col("different_starts") > 1)
        | (F.col("different_flags") > 1)
    )
)

if conflicts.limit(1).count():
    display(conflicts.limit(30))
    raise ValueError(
        "Actuals, starting dates, or imputation flags disagree across "
        "experiments. Select the intended experiment before proceeding."
    )

monthly_targets_df = (
    targets.groupBy(*target_keys)
    .agg(
        F.max("target_value").alias("target_value"),
        F.max("source_start_date").alias("source_start_date"),
        F.first(
            "target_is_imputed", ignorenulls=True
        ).alias("target_is_imputed"),
    )
    .withColumn(
        "observation_month",
        F.add_months(
            F.col("source_start_date"),
            F.col("month_index") - 1,
        ),
    )
)

# These start dates come from the existing curves. Their upstream
# construction must use the earliest sales/rental month respectively.
source_start_dates_df = (
    monthly_targets_df
    .groupBy("game_key")
    .agg(
        F.min(
            F.when(
                F.col("target_type") == "sales",
                F.col("source_start_date"),
            )
        ).alias("sales_start_date"),
        F.min(
            F.when(
                F.col("target_type") == "rental",
                F.col("source_start_date"),
            )
        ).alias("rental_start_date"),
    )
)

ai_attr_clean_df = ai_attr_clean_df.join(
    source_start_dates_df, "game_key", "left"
)


# ============================================================
# 4. Create the configured prediction months for every AI game
# ============================================================
prediction_grid = (
    ai_attr_clean_df
    .withColumn("month_index", F.explode(F.sequence(F.lit(1), F.lit(FORECAST_MONTHS))))
    .withColumn(
        "target_type",
        F.explode(F.array(F.lit("sales"), F.lit("rental"))),
    )
)

model_input_df = prediction_grid.join(
    monthly_targets_df,
    ["game_key", "target_type", "month_index"],
    "left",
)

# Positive identification of observed targets.
# Unknown imputation status is excluded until reviewed.
usable_actual = (
    F.col("target_value").isNotNull()
    & ~F.isnan("target_value")
    & (F.abs(F.col("target_value")) != F.lit(float("inf")))
    & (F.col("target_is_imputed") == F.lit(False))
)

sales_training_df = model_input_df.where(
    (F.col("target_type") == "sales") & usable_actual
)

rental_training_df = model_input_df.where(
    (F.col("target_type") == "rental") & usable_actual
)

# Explicit allowlist: names, dates, targets, match flags, and experiment
# metadata must not accidentally become predictors.
numeric_features = numeric_features + ["month_index"]
feature_columns = numeric_features + categorical_features

# Targets without matching AI keys: inspect rather than silently ignore.
unmatched_targets_df = monthly_targets_df.join(
    ai_attr_clean_df.select("game_key"),
    "game_key",
    "left_anti",
)

print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {categorical_features}")
print("Datasets: ai_attr_clean_df, model_input_df, "
      "sales_training_df, rental_training_df")

display(
    model_input_df
    .groupBy("target_type")
    .agg(
        F.countDistinct("game_key").alias("all_games"),
        F.countDistinct(
            F.when(usable_actual, F.col("game_key"))
        ).alias("games_with_observed_targets"),
        F.sum(
            F.when(usable_actual, 1).otherwise(0)
        ).alias("observed_training_rows"),
    )
)

display(unmatched_targets_df.limit(20))
display(ai_attr_clean_df.limit(10))

In [0]:
# ============================================================
# CELL 4 — Preprocessing, train/test split, tuning, and XGBoost models
# ============================================================
import uuid
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

try:
    from xgboost import XGBRegressor
except ImportError as exc:
    raise ImportError(
        "Run %pip install xgboost in a separate cell. "
        "Restart Python if requested, then rerun cells 1–4."
    ) from exc


# ============================================================
# 1. Configuration
# ============================================================
EXPERIMENT_NAME = f"xgb_ai_attributes_h{FORECAST_MONTHS}_tuned_v4"
EXPERIMENT_ID = "xgb_" + uuid.uuid4().hex[:12]


# ============================================================
# 2. Convert the prepared modeling dataset to pandas
# ============================================================
metadata = [
    "game_key",
    "game_name",
    "target_type",
    "target_value",
    "target_is_imputed",
    "source_start_date",
    "observation_month",
    "game_release_date",
    "sales_start_date",
    "rental_start_date",
    "to_sales_name_match",
    "to_rental_name_match",
]

model_pdf = model_input_df.select(
    *list(dict.fromkeys(metadata + feature_columns))
).toPandas()

for column in numeric_features + ["target_value"]:
    model_pdf[column] = (
        pd.to_numeric(model_pdf[column], errors="coerce")
        .astype(float)
        .replace([np.inf, -np.inf], np.nan)
    )

for column in categorical_features:
    model_pdf[column] = (
        model_pdf[column]
        .fillna("__missing__")
        .astype(str)
    )

date_columns = [
    "game_release_date",
    "sales_start_date",
    "rental_start_date",
    "source_start_date",
    "observation_month",
]

for column in date_columns:
    model_pdf[column] = pd.to_datetime(
        model_pdf[column],
        errors="coerce",
    )

# Only observed, non-imputed targets can enter training/evaluation.
model_pdf["is_observed"] = (
    model_pdf["target_value"].notna()
    & model_pdf["target_is_imputed"].eq(False).fillna(False)
)


# ============================================================
# 3. Shared chronological split by GAME
# ============================================================
# Each game's sales and rental rows receive the same split.
# Each source retains its own curve start date and month index.
game_dates = (
    model_pdf.groupby("game_key")["game_release_date"]
    .min()
)

commercial_dates = (
    model_pdf.loc[model_pdf["is_observed"]]
    .groupby("game_key")["source_start_date"]
    .min()
)

game_split_pdf = pd.concat(
    [
        game_dates.rename("metadata_release_date"),
        commercial_dates.rename("commercial_start_date"),
    ],
    axis=1,
)

game_split_pdf["first_observed_date"] = (
    game_split_pdf.min(axis=1)
)

labeled_games = set(
    model_pdf.loc[model_pdf["is_observed"], "game_key"]
)

eligible = (
    game_split_pdf.loc[
        game_split_pdf.index.isin(labeled_games)
        & game_split_pdf["first_observed_date"].notna()
    ]
    .sort_values("first_observed_date")
)

if len(eligible) < 10:
    raise ValueError(
        "Fewer than 10 dated games with observed targets."
    )

test_count = max(
    1,
    int(np.ceil(TEST_FRACTION * len(eligible))),
)

TEST_CUTOFF = eligible.iloc[-test_count]["first_observed_date"]

game_split_pdf["split"] = "predict_only"

game_split_pdf.loc[
    game_split_pdf.index.isin(labeled_games),
    "split",
] = "undated"

game_split_pdf.loc[eligible.index, "split"] = np.where(
    eligible["first_observed_date"] < TEST_CUTOFF,
    "train",
    "test",
)

# Games tied on the cutoff date stay together.
# Therefore, the actual split may differ slightly from 80/20.
model_pdf = model_pdf.join(
    game_split_pdf[["first_observed_date", "split"]],
    on="game_key",
)


# ============================================================
# 4. Check for shared source curves crossing train/test
# ============================================================
for target_type, mapping in [
    ("sales", "to_sales_name_match"),
    ("rental", "to_rental_name_match"),
]:
    aliases = (
        model_pdf.loc[
            model_pdf["target_type"].eq(target_type)
            & model_pdf["is_observed"]
            & model_pdf["split"].isin(["train", "test"]),
            [mapping, "game_key", "split"],
        ]
        .drop_duplicates()
        .copy()
    )

    aliases[mapping] = (
        aliases[mapping]
        .astype("string")
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
    )

    aliases = aliases.loc[
        aliases[mapping].notna()
        & ~aliases[mapping].isin(
            ["", "none", "null", "nan", "na", "n/a"]
        )
    ]

    overlap = aliases.groupby(mapping)["split"].nunique()
    overlap = overlap.index[overlap > 1]

    if len(overlap):
        display(
            aliases.loc[aliases[mapping].isin(overlap)]
        )
        raise ValueError(
            f"{target_type}: shared source curves cross train/test. "
            "Resolve these mappings or group the aliases before training."
        )


# ============================================================
# 5. Preprocessing helpers
# ============================================================
class UpperQuantileCapper(BaseEstimator, TransformerMixin):
    """Learn upper caps for reels and paylines/ways."""

    def __init__(
        self,
        columns=("reels", "paylines_or_ways"),
        quantile=0.99,
    ):
        self.columns = columns
        self.quantile = quantile

    def fit(self, X, y=None):
        self.feature_names_in_ = np.asarray(
            X.columns,
            dtype=object,
        )

        self.caps_ = {
            column: float(X[column].quantile(self.quantile))
            for column in self.columns
            if column in X.columns
        }

        return self

    def transform(self, X):
        result = X.copy()

        for column, cap in self.caps_.items():
            if np.isfinite(cap):
                result[column] = result[column].clip(
                    upper=cap
                )

        return result

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            return self.feature_names_in_
        return np.asarray(input_features)


def metric_record(frame, prediction_column):
    errors = (
        frame[prediction_column].to_numpy()
        - frame["target_value"].to_numpy()
    )

    return {
        "games": frame["game_key"].nunique(),
        "months": len(frame),
        "MAE": float(np.mean(np.abs(errors))),
        "RMSE": float(np.sqrt(np.mean(errors ** 2))),
        "Bias": float(np.mean(errors)),
    }


# ============================================================
# Hyperparameter tuning inside the OUTER training split only
# ============================================================
from sklearn.model_selection import ParameterSampler
import json
import copy


def build_preprocessor(fit_features):
    num_cols = [c for c in numeric_features if fit_features[c].notna().any()]
    dropped = sorted(set(numeric_features) - set(num_cols))
    try:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)
    preprocessor = ColumnTransformer([
        ("numeric", Pipeline([
            ("cap", UpperQuantileCapper(quantile=CAP_QUANTILE)),
            ("impute", SimpleImputer(strategy="median", add_indicator=True)),
            ("scale", MinMaxScaler(clip=True)),
        ]), num_cols),
        ("category", encoder, categorical_features),
    ])
    preprocessor.fit(fit_features[feature_columns])
    return preprocessor, dropped, list(preprocessor.get_feature_names_out())


def sales_constraints(training, output_features):
    increasing = bool(
        training["target_type"].eq("sales").all()
        and training["target_value"].ge(0).all()
        and not training.sort_values(["game_key", "month_index"])
        .groupby("game_key")["target_value"].diff().lt(-1e-8).any()
    )
    constraints = [0] * len(output_features)
    if increasing:
        constraints[output_features.index("numeric__month_index")] = 1
    return tuple(constraints), increasing


def equal_game_weights(frame):
    weights = 1.0 / frame.groupby("game_key")["game_key"].transform("size")
    return weights / weights.mean()


def make_tuning_folds(curve, training, target_type):
    # Source aliases stay together. The outer test games are absent from training.
    mapping = "to_sales_name_match" if target_type == "sales" else "to_rental_name_match"
    games = training[["game_key", "first_observed_date", mapping]].drop_duplicates("game_key").copy()
    source = (games[mapping].astype("string").str.strip().str.lower()
              .str.replace(r"\s+", " ", regex=True))
    source = source.mask(source.isin(["", "none", "null", "nan", "na", "n/a"]))
    games["source_group"] = ("source:" + source).fillna("game:" + games["game_key"].astype(str))
    games["cohort_date"] = games.groupby("source_group")["first_observed_date"].transform("min")
    dates = np.array(sorted(games["cohort_date"].dropna().unique()))
    warmup = max(1, int(np.ceil(len(dates) * TUNING_INITIAL_TRAIN_FRACTION)))
    date_blocks = np.array_split(dates[warmup:], TUNING_FOLDS)
    folds, audit, memberships = [], [], []
    for fold_number, dates_block in enumerate(date_blocks, 1):
        if len(dates_block) == 0:
            continue
        cutoff = pd.Timestamp(dates_block[0])
        fit_keys = set(games.loc[games["cohort_date"].lt(cutoff)
                                 & games["first_observed_date"].lt(cutoff), "game_key"])
        valid_keys = set(games.loc[games["cohort_date"].isin(dates_block), "game_key"])
        # Even an old training game's later outcomes must not pass the fold cutoff.
        fit = training.loc[training["game_key"].isin(fit_keys)
                           & training["observation_month"].lt(cutoff)].copy()
        valid = training.loc[training["game_key"].isin(valid_keys)
                             & training["observation_month"].ge(cutoff)
                             & training["observation_month"].lt(TEST_CUTOFF)].copy()
        fit_sources = set(games.loc[games["game_key"].isin(fit["game_key"]), "source_group"])
        valid_sources = set(games.loc[games["game_key"].isin(valid["game_key"]), "source_group"])
        usable = (len(fit_sources) >= TUNING_MIN_TRAIN_GROUPS
                  and len(valid_sources) >= TUNING_MIN_VALIDATION_GROUPS)
        record = {
            "target_type": target_type, "fold": fold_number, "cutoff": cutoff,
            "last_validation_cohort_date": pd.Timestamp(dates_block[-1]),
            "training_games": fit["game_key"].nunique(), "validation_games": valid["game_key"].nunique(),
            "training_source_groups": len(fit_sources), "validation_source_groups": len(valid_sources),
            "training_rows": len(fit), "validation_rows": len(valid),
            "latest_training_target_month": fit["observation_month"].max(),
            "latest_validation_target_month": valid["observation_month"].max(),
            "status": "used" if usable else "insufficient_history",
        }
        audit.append(record)
        if not usable:
            continue
        assert not fit_sources & valid_sources
        assert not set(fit["game_key"]) & set(valid["game_key"])
        assert fit["split"].eq("train").all() and valid["split"].eq("train").all()
        assert fit["observation_month"].lt(cutoff).all()
        assert valid["observation_month"].lt(TEST_CUTOFF).all()
        # Known horizons may use the full feature grid, but only for fit games.
        fit_grid = curve.loc[curve["game_key"].isin(fit["game_key"])]
        preprocessor, _, names = build_preprocessor(fit_grid)
        constraints, _ = sales_constraints(fit, names)
        folds.append({
            "fold": fold_number, "fit_x": preprocessor.transform(fit[feature_columns]),
            "fit_y": fit["target_value"].to_numpy(), "fit_weights": equal_game_weights(fit).to_numpy(),
            "valid_x": preprocessor.transform(valid[feature_columns]),
            "valid_y": valid["target_value"].to_numpy(), "valid_games": valid["game_key"].to_numpy(),
            "constraints": constraints, "clip_at_zero": bool(fit["target_value"].ge(0).all()),
        })
        for role, sample in [("train", fit), ("validation", valid)]:
            members = sample.groupby("game_key").agg(
                observed_rows=("month_index", "size"),
                first_target_month=("observation_month", "min"),
                last_target_month=("observation_month", "max"),
            ).reset_index().merge(games[["game_key", "source_group", "cohort_date"]], on="game_key", validate="one_to_one")
            members["target_type"] = target_type
            members["fold"] = fold_number
            members["fold_cutoff"] = cutoff
            members["role"] = role
            memberships.extend(members.to_dict("records"))
    return folds, audit, memberships


def tune_model(curve, training, target_type):
    base = dict(XGB_PARAMS)
    summary = {"target_type": target_type, "status": "disabled", "selected_trial": None,
               "folds": 0, "trials": 0, "metric": TUNING_METRIC,
               "default_validation_score": None, "best_validation_score": None,
               "validation_improvement_pct": None}
    if not TUNE_HYPERPARAMETERS:
        return base, summary
    folds, audit, membership = make_tuning_folds(curve, training, target_type)
    tuning_fold_rows.extend(audit)
    tuning_membership_rows.extend(membership)
    summary["folds"] = len(folds)
    if len(folds) < TUNING_MIN_FOLDS:
        summary["status"] = "insufficient_validation_history"
        print(f"{target_type}: only {len(folds)} usable chronological folds; retaining configured parameters.")
        return base, summary

    candidates = [base]
    seen = {json.dumps(base, sort_keys=True)}
    if TUNING_TRIALS > 1:
        for sampled in ParameterSampler(TUNING_SEARCH_SPACE, n_iter=TUNING_TRIALS - 1, random_state=RANDOM_SEED):
            params = {**base, **sampled}
            # Convert numpy scalar values to JSON-native values for durable logging.
            params = {k: v.item() if isinstance(v, np.generic) else v for k, v in params.items()}
            signature = json.dumps(params, sort_keys=True)
            if signature not in seen:
                seen.add(signature)
                candidates.append(params)
    scores = []
    for trial, params in enumerate(candidates):
        fold_scores = []
        for fold in folds:
            model = XGBRegressor(**params, monotone_constraints=fold["constraints"])
            model.fit(fold["fit_x"], fold["fit_y"], sample_weight=fold["fit_weights"])
            predictions = np.asarray(model.predict(fold["valid_x"]), dtype=float)
            if fold["clip_at_zero"]:
                predictions = np.maximum(predictions, 0)
            if not np.isfinite(predictions).all():
                raise ValueError(f"Non-finite tuning predictions: {target_type}, trial {trial}, fold {fold['fold']}.")
            error = predictions - fold["valid_y"]
            per_game = pd.DataFrame({"game_key": fold["valid_games"], "abs_error": np.abs(error),
                                     "squared_error": error ** 2, "error": error}).groupby("game_key").mean()
            metrics = {
                "target_type": target_type, "trial": trial, "fold": fold["fold"],
                "validation_games": len(per_game), "validation_rows": len(error),
                "game_weighted_mae": float(per_game["abs_error"].mean()),
                "game_weighted_mse": float(per_game["squared_error"].mean()),
                "game_weighted_bias": float(per_game["error"].mean()),
                "row_mae": float(np.mean(np.abs(error))), "row_mse": float(np.mean(error ** 2)),
            }
            fold_scores.append(metrics)
            tuning_trial_fold_rows.append(metrics)
        # Validation cohorts are disjoint. Weight folds by games to give each
        # validation game equal influence despite differing history lengths.
        game_weights = np.array([r["validation_games"] for r in fold_scores])
        row_weights = np.array([r["validation_rows"] for r in fold_scores])
        weighted = lambda field, weights: float(np.average([r[field] for r in fold_scores], weights=weights))
        row = {
            "target_type": target_type, "trial": trial, "is_configured_default": trial == 0,
            "folds": len(folds), "validation_games": int(game_weights.sum()),
            "validation_rows": int(row_weights.sum()),
            "game_weighted_mae": weighted("game_weighted_mae", game_weights),
            "game_weighted_rmse": float(np.sqrt(weighted("game_weighted_mse", game_weights))),
            "game_weighted_bias": weighted("game_weighted_bias", game_weights),
            "row_mae": weighted("row_mae", row_weights),
            "row_rmse": float(np.sqrt(weighted("row_mse", row_weights))),
            "params_json": json.dumps(params, sort_keys=True),
        }
        row["selection_score"] = row["game_weighted_" + TUNING_METRIC.lower()]
        scores.append(row)
        print(f"{target_type} trial {trial + 1}/{len(candidates)}: validation {TUNING_METRIC} = {row['selection_score']:.3f}")
    # The configured default wins an exact tie, so complexity does not win by accident.
    best = min(scores, key=lambda row: (row["selection_score"], row["trial"]))
    for row in scores:
        row["selected"] = row["trial"] == best["trial"]
        tuning_trial_rows.append(row)
    default_score = scores[0]["selection_score"]
    summary.update(status="tuned", selected_trial=best["trial"], trials=len(scores),
                   default_validation_score=default_score, best_validation_score=best["selection_score"],
                   validation_improvement_pct=(100 * (default_score - best["selection_score"]) / default_score
                                               if default_score > 0 else None))
    return candidates[best["trial"]], summary


# ============================================================
# 6. Train the two models
# ============================================================
xgb_models = {}
xgb_model_info = {}
best_params_by_target = {}
tuning_trial_rows, tuning_trial_fold_rows = [], []
tuning_fold_rows, tuning_membership_rows, tuning_summary_rows = [], [], []

prediction_frames = []
metric_rows = []
game_metric_rows = []
split_rows = []

for target_type in ["sales", "rental"]:
    curve = (
        model_pdf.loc[
            model_pdf["target_type"].eq(target_type)
        ]
        .copy()
        .sort_values(["game_key", "month_index"])
    )

    # Training actuals must be dated BEFORE the test cutoff,
    # including actuals from older training games.
    train_mask = (
        curve["split"].eq("train")
        & curve["is_observed"]
        & curve["observation_month"].lt(TEST_CUTOFF)
    )

    test_mask = (
        curve["split"].eq("test")
        & curve["is_observed"]
        & curve["observation_month"].ge(TEST_CUTOFF)
    )

    training = curve.loc[train_mask]
    test = curve.loc[test_mask]

    n_train = training["game_key"].nunique()
    n_test = test["game_key"].nunique()

    if n_train < 5 or n_test < 1:
        print(
            f"Skipping {target_type}: "
            f"{n_train} training / {n_test} test games."
        )
        continue

    # Fit preprocessing using only games with usable training rows.
    # Their static attributes + known month indices within the configured horizon are safe to use.
    # The complete grid gives each game equal preprocessing weight.
    fit_features = curve.loc[
        curve["game_key"].isin(training["game_key"])
    ]

    # Select parameters using only chronological folds inside the outer train split.
    selected_params, tuning_summary = tune_model(curve, training, target_type)
    best_params_by_target[target_type] = dict(selected_params)
    tuning_summary_rows.append(tuning_summary)

    # Refit preprocessing and the winning configuration on ALL outer training rows.
    preprocessing, dropped, output_features = build_preprocessor(fit_features)
    train_x = preprocessing.transform(training[feature_columns])
    constraints, sales_nondecreasing = sales_constraints(training, output_features)
    weights = equal_game_weights(training)
    model = XGBRegressor(**selected_params, monotone_constraints=constraints)

    model.fit(
        train_x,
        training["target_value"],
        sample_weight=weights,
    )

    fitted_pipeline = Pipeline(
        steps=[
            ("preprocess", preprocessing),
            ("model", model),
        ]
    )

    xgb_models[target_type] = fitted_pipeline

    # Predict all configured months for every game, including missing-target games.
    curve["prediction"] = fitted_pipeline.predict(
        curve[feature_columns]
    )

    clip_at_zero = bool(
        training["target_value"].ge(0).all()
    )

    if clip_at_zero:
        curve["prediction"] = curve["prediction"].clip(
            lower=0
        )

    # --------------------------------------------------------
    # Simple baseline: weighted training mean for each month
    # --------------------------------------------------------
    baseline_input = training.assign(
        _weight=weights,
        _weighted_target=training["target_value"] * weights,
    )

    by_month = (
        baseline_input.groupby("month_index")[
            ["_weight", "_weighted_target"]
        ]
        .sum()
    )

    baseline = (
        by_month["_weighted_target"]
        / by_month["_weight"]
    )

    fallback = float(
        np.average(
            training["target_value"],
            weights=weights,
        )
    )

    curve["baseline_prediction"] = (
        curve["month_index"]
        .map(baseline)
        .fillna(fallback)
    )

    curve["used_for_fit"] = train_mask
    curve["used_for_test"] = test_mask
    curve["experiment_id"] = EXPERIMENT_ID
    curve["experiment_name"] = EXPERIMENT_NAME

    prediction_frames.append(curve)

    # --------------------------------------------------------
    # Evaluate only observed test targets
    # --------------------------------------------------------
    evaluation = curve.loc[test_mask]

    for estimator, prediction_column in [
        ("xgboost", "prediction"),
        ("monthly_mean", "baseline_prediction"),
    ]:
        metric_rows.append(
            {
                "target_type": target_type,
                "estimator": estimator,
                **metric_record(
                    evaluation,
                    prediction_column,
                ),
            }
        )

    for game_key, group in evaluation.groupby("game_key"):
        game_metric_rows.append(
            {
                "target_type": target_type,
                "game_key": game_key,
                **metric_record(group, "prediction"),
            }
        )

    # Retain preprocessing/model information for later prediction.
    xgb_model_info[target_type] = {
        "experiment_id": EXPERIMENT_ID,
        "experiment_name": EXPERIMENT_NAME,
        "test_cutoff": TEST_CUTOFF,
        "feature_columns": list(feature_columns),
        "dropped_empty_features": dropped,
        "transformed_features": output_features,
        "sales_month_monotonic": sales_nondecreasing,
        "clip_predictions_at_zero": clip_at_zero,
        "selected_params": dict(selected_params),
        "tuning": dict(tuning_summary),
        "upper_caps": (
            preprocessing
            .named_transformers_["numeric"]
            .named_steps["cap"]
            .caps_
        ),
    }

    split_rows.append(
        {
            "target_type": target_type,
            "training_games": n_train,
            "test_games": n_test,
            "training_months": len(training),
            "test_months": len(test),
            "full_horizon_test_games": int(
                evaluation.groupby("game_key")
                .size()
                .eq(FORECAST_MONTHS)
                .sum()
            ),
            "post_cutoff_training_rows_excluded": int(
                (
                    curve["split"].eq("train")
                    & curve["is_observed"]
                    & curve["observation_month"].ge(TEST_CUTOFF)
                ).sum()
            ),
        }
    )

    print(
        f"{target_type}: fitted on {n_train} games; "
        f"test on {n_test} games."
    )
    print(f"Dropped all-missing numeric features: {dropped}")


# ============================================================
# 7. Results
# ============================================================
if not prediction_frames:
    raise ValueError(
        "Neither model had enough dated training/test observations."
    )

xgb_predictions_pdf = pd.concat(
    prediction_frames,
    ignore_index=True,
)

xgb_metrics_pdf = pd.DataFrame(metric_rows)
xgb_game_metrics_pdf = pd.DataFrame(game_metric_rows)
xgb_split_summary_pdf = pd.DataFrame(split_rows)
tuning_summary_pdf = pd.DataFrame(tuning_summary_rows)
tuning_trials_pdf = pd.DataFrame(tuning_trial_rows)
tuning_trial_folds_pdf = pd.DataFrame(tuning_trial_fold_rows)
tuning_folds_pdf = pd.DataFrame(tuning_fold_rows)
tuning_membership_pdf = pd.DataFrame(tuning_membership_rows)
for frame in [tuning_summary_pdf, tuning_trials_pdf, tuning_trial_folds_pdf,
              tuning_folds_pdf, tuning_membership_pdf]:
    frame["experiment_id"] = EXPERIMENT_ID
    frame["experiment_name"] = EXPERIMENT_NAME
print("Tuning status and validation improvement over the configured default:")
display(tuning_summary_pdf)
if not tuning_trials_pdf.empty:
    display(tuning_trials_pdf.sort_values(["target_type", "selection_score", "trial"]))
if not tuning_folds_pdf.empty:
    display(tuning_folds_pdf)

for result in [xgb_metrics_pdf, xgb_game_metrics_pdf]:
    result["experiment_id"] = EXPERIMENT_ID
    result["experiment_name"] = EXPERIMENT_NAME

print(f"\nExperiment: {EXPERIMENT_ID}")
print(f"Shared test cutoff: {TEST_CUTOFF.date()}")
print("Sales target: cumulative units.")
print("Rental target: monthly units.")
print("Bias = prediction - actual; positive means overprediction.")
print("Metrics use observed test months only.")
print("Models are available in xgb_models.")
print(f"All {FORECAST_MONTHS}-month predictions are in xgb_predictions_pdf.")

display(xgb_split_summary_pdf)

display(xgb_metrics_pdf)

display(
    xgb_game_metrics_pdf.sort_values(
        ["target_type", "RMSE"],
        ascending=[True, False],
    )
)

# Capture the configuration used by these fitted models for MLflow provenance.
BASELINE_TRAINING_CONFIG = {
    "FORECAST_MONTHS": FORECAST_MONTHS, "TEST_FRACTION": TEST_FRACTION,
    "CAP_QUANTILE": CAP_QUANTILE, "RANDOM_SEED": RANDOM_SEED,
    "XGB_PARAMS": copy.deepcopy(XGB_PARAMS), "EXPERIMENT_ID": EXPERIMENT_ID,
    **{name: copy.deepcopy(globals()[name]) for name in [
        "TUNE_HYPERPARAMETERS", "TUNING_TRIALS", "TUNING_FOLDS", "TUNING_MIN_FOLDS",
        "TUNING_INITIAL_TRAIN_FRACTION", "TUNING_MIN_TRAIN_GROUPS",
        "TUNING_MIN_VALIDATION_GROUPS", "TUNING_METRIC", "TUNING_SEARCH_SPACE",
    ]},
}


# Compare accuracy at each forecast month using only observed test targets.
monthly_metric_rows = []
test_predictions = xgb_predictions_pdf.loc[xgb_predictions_pdf["used_for_test"]]
for (target_type, month), rows in test_predictions.groupby(["target_type", "month_index"]):
    for estimator, column in [("xgboost", "prediction"), ("monthly_mean", "baseline_prediction")]:
        monthly_metric_rows.append({
            "target_type": target_type, "month_index": int(month),
            "window": f"month_{int(month):02d}", "estimator": estimator,
            **metric_record(rows, column),
            "experiment_id": EXPERIMENT_ID, "experiment_name": EXPERIMENT_NAME,
        })
xgb_monthly_metrics_pdf = pd.DataFrame(monthly_metric_rows)
display(xgb_monthly_metrics_pdf)


In [0]:
# ============================================================
# CELL 5 — Select a game, plot predictions, inspect SHAP values
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import textwrap

from xgboost import DMatrix

TEST_ONLY = True
TOP_FEATURES = 15


# ============================================================
# 1. Select game/model and month to explain
# ============================================================
eligible = xgb_predictions_pdf.loc[
    xgb_predictions_pdf["target_type"].isin(xgb_models)
].copy()

if TEST_ONLY:
    eligible = eligible.loc[eligible["split"].eq("test")]
    valid_pairs = eligible.loc[eligible["used_for_test"], ["game_key", "target_type"]].drop_duplicates()
    eligible = eligible.merge(valid_pairs, on=["game_key", "target_type"], validate="many_to_one")

game_list = (
    eligible[["game_key", "target_type", "split"]]
    .drop_duplicates()
    .sort_values(["game_key", "target_type"])
)

if game_list.empty:
    raise ValueError(
        "No selectable games. Run cell 4 or set TEST_ONLY = False."
    )

labels = {"sales": "Sale", "rental": "Rent"}

game_options = {
    f"{r.game_key} | {labels[r.target_type]} | {r.split}": (
        r.game_key,
        r.target_type,
    )
    for r in game_list.itertuples(index=False)
}


def dropdown_value(name, choices, label, default=None):
    default = default or choices[0]

    try:
        previous = dbutils.widgets.get(name)
        if previous in choices:
            default = previous
    except Exception:
        pass  # First run: widget does not exist yet.

    dbutils.widgets.dropdown(name, default, choices, label)
    selected = dbutils.widgets.get(name)

    if selected not in choices:
        raise ValueError(
            f"Choose a current value for {label}, then rerun this cell."
        )

    return selected


selected_label = dropdown_value(
    "xgb_game_shap",
    list(game_options),
    "Game / model",
)

selected_month = int(
    dropdown_value(
        "xgb_shap_month",
        [str(i) for i in range(1, FORECAST_MONTHS + 1)],
        "Month to explain",
        default=str(FORECAST_MONTHS),
    )
)

selected_game, target_type = game_options[selected_label]

rows = (
    eligible.loc[
        eligible["game_key"].eq(selected_game)
        & eligible["target_type"].eq(target_type)
    ]
    .sort_values("month_index")
    .reset_index(drop=True)
)

if sorted(rows["month_index"].astype(int).tolist()) != list(range(1, FORECAST_MONTHS + 1)):
    raise ValueError(
        f"Expected exactly {FORECAST_MONTHS} rows for this game/model. Rerun cell 4."
    )


# ============================================================
# 2. Apply the existing fitted preprocessing and model
# ============================================================
pipeline = xgb_models[target_type]
info = xgb_model_info[target_type]

preprocessor = pipeline.named_steps["preprocess"]
model = pipeline.named_steps["model"]
booster = model.get_booster()

# Transform only: do not refit preprocessing.
X = preprocessor.transform(rows[info["feature_columns"]])
feature_names = list(preprocessor.get_feature_names_out())

dmatrix = DMatrix(
    X,
    feature_names=booster.feature_names,
    missing=model.get_params().get("missing", np.nan),
)

# Match the model's tree range if early stopping is added later.
best_iteration = getattr(model, "best_iteration", None)

tree_range = (
    {}
    if best_iteration is None
    else {"iteration_range": (0, best_iteration + 1)}
)


# ============================================================
# 3. Calculate SHAP contributions for all configured months
# ============================================================
contributions = np.asarray(
    booster.predict(
        dmatrix,
        pred_contribs=True,
        approx_contribs=False,
        **tree_range,
    )
)

if contributions.ndim == 3 and contributions.shape[1] == 1:
    contributions = contributions[:, 0, :]

if contributions.shape != (len(rows), len(feature_names) + 1):
    raise ValueError(
        "Unexpected SHAP shape. Expected single-output regression."
    )

# Final column = SHAP base value.
shap_values = contributions[:, :-1]
base_values = contributions[:, -1]

raw_predictions = np.asarray(model.predict(X)).reshape(-1)

# Cell 4 uses squared-error regression, so contributions are in units.
reconstructed = base_values + shap_values.sum(axis=1)

tolerance = 1e-4 * np.maximum(
    1.0,
    np.abs(contributions).sum(axis=1),
)

if np.any(np.abs(reconstructed - raw_predictions) > tolerance):
    raise ValueError(
        "SHAP values do not reconstruct this model's raw predictions."
    )

# Apply the same prediction floor as cell 4.
predictions = (
    np.maximum(raw_predictions, 0)
    if info.get("clip_predictions_at_zero", False)
    else raw_predictions
)

position = int(
    np.flatnonzero(rows["month_index"].eq(selected_month))[0]
)

feature_values = X[position:position + 1]

if hasattr(feature_values, "toarray"):
    feature_values = feature_values.toarray()

# Keep all configured months of SHAP values for further analysis.
selected_game_shap_pdf = pd.DataFrame(
    shap_values,
    columns=feature_names,
)

selected_game_shap_pdf.insert(
    0,
    "month_index",
    rows["month_index"].astype(int),
)

selected_game_shap_pdf["base_value"] = base_values
selected_game_shap_pdf["raw_prediction"] = raw_predictions
selected_game_shap_pdf["prediction"] = predictions

# Detailed explanation for the selected month.
selected_month_shap_pdf = pd.DataFrame(
    {
        "feature": feature_names,
        "model_feature_value": np.asarray(feature_values).reshape(-1),
        "shap_units": shap_values[position],
    }
)

selected_month_shap_pdf["abs_shap"] = (
    selected_month_shap_pdf["shap_units"].abs()
)

selected_month_shap_pdf = (
    selected_month_shap_pdf
    .sort_values("abs_shap", ascending=False)
    .reset_index(drop=True)
)


# ============================================================
# 4. Prepare the most influential features
# ============================================================
top = (
    selected_month_shap_pdf
    .head(TOP_FEATURES)[["feature", "shap_units"]]
    .copy()
)

remaining = selected_month_shap_pdf.iloc[TOP_FEATURES:]

if not remaining.empty:
    top.loc[len(top)] = [
        "Other features (sum)",
        remaining["shap_units"].sum(),
    ]


def readable_feature(name):
    name = (
        name.removeprefix("numeric__")
        .removeprefix("category__")
    )

    return textwrap.fill(
        name.replace("_", " "),
        width=34,
    )


# ============================================================
# 5. Plot time series and SHAP contributions
# ============================================================
fig, (ax_curve, ax_shap) = plt.subplots(
    1,
    2,
    figsize=(19, 9),
    gridspec_kw={"width_ratios": [1.15, 1]},
    constrained_layout=True,
)

months = rows["month_index"].astype(int)

# Missing/imputed actuals stay hidden.
actuals = rows["target_value"].where(rows["is_observed"])

units = (
    "Cumulative sales units"
    if target_type == "sales"
    else "Monthly rental units"
)

ax_curve.plot(
    months,
    predictions,
    "o-",
    color="#006BB6",
    label="XGBoost",
)

ax_curve.plot(
    months,
    actuals,
    "o-",
    color="#E57A21",
    label="Observed actuals",
)

ax_curve.plot(
    months,
    rows["baseline_prediction"],
    "--",
    color="#888888",
    label="Training mean baseline",
)

ax_curve.axvline(
    selected_month,
    color="#333333",
    alpha=0.35,
    linestyle=":",
)

ax_curve.scatter(
    [selected_month],
    [predictions[position]],
    s=130,
    facecolors="none",
    edgecolors="#006BB6",
    linewidths=2,
    zorder=5,
)

ax_curve.set(
    title=(
        f"{selected_game}\n"
        f"{labels[target_type]} | {rows['split'].iloc[0]}"
    ),
    xlabel="Month from source start",
    ylabel=units,
    xticks=range(1, FORECAST_MONTHS + 1),
)

ax_curve.grid(alpha=0.2)
ax_curve.legend()

# Signed SHAP values: positive increases the prediction.
y = np.arange(len(top))

ax_shap.barh(
    y,
    top["shap_units"],
    color=np.where(
        top["shap_units"] >= 0,
        "#006BB6",
        "#E57A21",
    ),
)

ax_shap.set_yticks(y)

ax_shap.set_yticklabels(
    [readable_feature(c) for c in top["feature"]],
    fontsize=9,
)

ax_shap.invert_yaxis()
ax_shap.axvline(0, color="#444444", linewidth=0.8)

ax_shap.set(
    title=(
        f"SHAP explanation — month {selected_month}\n"
        "Contribution to raw model prediction"
    ),
    xlabel="Contribution in units (+ increases; − decreases)",
)

ax_shap.grid(axis="x", alpha=0.2)

plt.show()


# ============================================================
# 6. Explanation and detailed feature values
# ============================================================
actual_text = (
    f"{actuals.iloc[position]:,.1f}"
    if pd.notna(actuals.iloc[position])
    else "unavailable"
)

print(
    f"Month {selected_month}: "
    f"prediction = {predictions[position]:,.1f}; "
    f"actual = {actual_text}"
)

print(
    f"SHAP base {base_values[position]:,.1f} "
    f"+ contributions {shap_values[position].sum():+,.1f} "
    f"= raw prediction {raw_predictions[position]:,.1f}"
)

if not np.isclose(
    raw_predictions[position],
    predictions[position],
):
    print(
        "The displayed prediction is floored at zero. "
        "SHAP explains the raw prediction before that floor."
    )

print(
    "SHAP explains this model's prediction; "
    "these contributions are not causal effects."
)

print(
    "model_feature_value contains the encoded/scaled "
    "value used by the model."
)

display(
    selected_month_shap_pdf.drop(columns="abs_shap")
)

In [0]:
# CELL 6 — MLflow tracking for the fitted sales and rental forecasts
# Run cells 1–4 first. Cell 5 is an optional forecast/SHAP viewer.
# Each execution logs parent -> final target models -> validation trial runs.
# Trial results are recorded here after cell 4 finishes tuning and refitting.
import inspect
import tempfile
from pathlib import Path
from importlib.metadata import version
from sklearn.base import BaseEstimator, RegressorMixin

try:
    import mlflow
    import mlflow.sklearn
    import mlflow.pyfunc
    from mlflow.models import infer_signature
except ImportError as exc:
    raise ImportError("Install mlflow in Databricks, then rerun this cell.") from exc


class ForecastModel(RegressorMixin, BaseEstimator):
    """Clean game attributes plus month_index -> final demand forecast."""
    def __init__(self, pipeline, numeric, categorical, horizon, clip_at_zero):
        self.pipeline = pipeline
        self.numeric = numeric
        self.categorical = categorical
        self.horizon = horizon
        self.clip_at_zero = clip_at_zero

    def __sklearn_is_fitted__(self):
        return True

    def predict(self, X):
        data = pd.DataFrame(X).copy()
        columns = self.numeric + self.categorical
        missing = set(columns) - set(data.columns)
        if missing:
            raise ValueError(f"Missing cleaned input columns: {sorted(missing)}")
        for column in self.numeric:
            data[column] = pd.to_numeric(data[column], errors="coerce").replace([np.inf, -np.inf], np.nan)
        for column in self.categorical:
            data[column] = data[column].fillna("__missing__").astype(str)
        month = data["month_index"]
        if not (month.notna() & month.between(1, self.horizon) & month.eq(month.round())).all():
            raise ValueError(f"month_index must be an integer from 1 to {self.horizon}.")
        if data.empty:
            return np.array([], dtype=float)
        result = self.pipeline.predict(data[columns])
        return np.maximum(result, 0) if self.clip_at_zero else result


def log_frame(frame, name):
    with tempfile.TemporaryDirectory(prefix="demand_mlflow_") as directory:
        path = Path(directory) / f"{name}.csv.gz"
        frame.to_csv(path, index=False, compression="gzip")
        mlflow.log_artifact(str(path), artifact_path="tables")


def log_scores(frame, prefix):
    for row in frame.to_dict("records"):
        window = row.get("window", "all_months")
        metrics = {f"{prefix}/{window}/{name}": float(row[name])
                   for name in ["MAE", "RMSE", "Bias", "games", "months"]
                   if np.isfinite(row[name])}
        if metrics:
            mlflow.log_metrics(metrics)


if "BASELINE_TRAINING_CONFIG" not in globals():
    raise ValueError("Run cells 1–4 before logging to MLflow.")
for name, fitted_value in BASELINE_TRAINING_CONFIG.items():
    if name in globals() and globals()[name] != fitted_value:
        raise ValueError(f"{name} changed after training. Rerun cells 1–4 before logging.")

experiment_path = MLFLOW_EXPERIMENT_PATH
if not experiment_path:
    current_user = spark.sql("SELECT current_user() AS user_name").first()["user_name"]
    experiment_path = f"/Users/{current_user}/AI_feature_extraction_forecasts"
if not experiment_path.startswith("/"):
    raise ValueError("MLFLOW_EXPERIMENT_PATH must be an absolute Databricks experiment path.")
experiment = mlflow.set_experiment(experiment_path)
MLFLOW_EXPERIMENT_ID = experiment.experiment_id
requirements = [f"mlflow=={mlflow.__version__}"] + [
    f"{package}=={version(package)}"
    for package in ["cloudpickle", "numpy", "pandas", "scipy", "scikit-learn", "xgboost"]
]
mlflow_forecast_models, run_records, tuning_run_records = {}, [], []

with mlflow.start_run(
    experiment_id=MLFLOW_EXPERIMENT_ID,
    run_name=MLFLOW_RUN_NAME or EXPERIMENT_NAME,
    nested=mlflow.active_run() is not None,
    tags={"project": "AI feature extraction", "forecast_origin": "prelaunch",
          "model_family": "attributes_only", "experiment_id": EXPERIMENT_ID},
) as parent:
    MLFLOW_PARENT_RUN_ID = parent.info.run_id
    mlflow.log_params({
        "forecast_months": FORECAST_MONTHS, "test_fraction": TEST_FRACTION,
        "test_cutoff": str(TEST_CUTOFF.date()), "random_seed": RANDOM_SEED,
        "cap_quantile": CAP_QUANTILE,
        "tuning_enabled": TUNE_HYPERPARAMETERS, "tuning_trials_requested": TUNING_TRIALS,
        "tuning_folds_requested": TUNING_FOLDS, "tuning_metric": TUNING_METRIC,
        "tuning_weighting": "equal_weight_per_validation_game",
        "sales_target": "cumulative_units", "rental_target": "monthly_units",
        **{"base_xgb/" + k: v for k, v in BASELINE_TRAINING_CONFIG["XGB_PARAMS"].items()},
    })
    mlflow.log_dict({
        "training_config": BASELINE_TRAINING_CONFIG,
        "source_tables": SOURCE_TABLES,
        "selected_parameters": best_params_by_target,
        "tuning_method": "Random search with default candidate; expanding chronological game/source-group folds inside training only.",
        "numeric_features": numeric_features, "categorical_features": categorical_features,
        "requirements": requirements,
        "data_snapshot_logged": MLFLOW_LOG_DATA_SNAPSHOT,
        "evaluation": "Fixed historical cutoff; model fitted on earlier games and pre-cutoff observed targets.",
    }, "configuration.json")

    for target_type, pipeline in xgb_models.items():
        with mlflow.start_run(
            experiment_id=MLFLOW_EXPERIMENT_ID, run_name=f"xgb_{target_type}", nested=True,
            tags={"target_type": target_type, "model_family": "attributes_only"},
        ) as child:
            info = xgb_model_info[target_type]
            summary = info["tuning"]
            mlflow.log_params({"tuning/status": summary["status"],
                               "tuning/trials": summary["trials"],
                               "tuning/folds": summary["folds"],
                               "tuning/metric": summary["metric"],
                               "tuning/selected_trial": str(summary["selected_trial"])})
            mlflow.log_dict(info["selected_params"], "selected_parameters.json")
            validation_scores = {f"validation/{key}": float(summary[key])
                                 for key in ["default_validation_score", "best_validation_score",
                                             "validation_improvement_pct"]
                                 if summary[key] is not None and np.isfinite(summary[key])}
            if validation_scores:
                mlflow.log_metrics(validation_scores)

            if not tuning_trials_pdf.empty:
                trials = tuning_trials_pdf.loc[tuning_trials_pdf["target_type"].eq(target_type)]
                for trial in trials.to_dict("records"):
                    with mlflow.start_run(
                        experiment_id=MLFLOW_EXPERIMENT_ID,
                        run_name=f"{target_type}_trial_{int(trial['trial']):03d}", nested=True,
                        tags={"target_type": target_type, "run_role": "tuning_trial",
                              "selected": str(bool(trial["selected"])).lower(),
                              "experiment_id": EXPERIMENT_ID},
                    ) as trial_run:
                        mlflow.log_params(json.loads(trial["params_json"]))
                        mlflow.log_params({"selection_metric": TUNING_METRIC,
                                           "is_configured_default": trial["is_configured_default"],
                                           "folds": trial["folds"],
                                           "validation_games": trial["validation_games"],
                                           "validation_rows": trial["validation_rows"]})
                        mlflow.log_metrics({f"validation/{name}": float(trial[name]) for name in [
                            "selection_score", "game_weighted_mae", "game_weighted_rmse",
                            "game_weighted_bias", "row_mae", "row_rmse",
                        ]})
                        trial_folds = tuning_trial_folds_pdf.loc[
                            tuning_trial_folds_pdf["target_type"].eq(target_type)
                            & tuning_trial_folds_pdf["trial"].eq(trial["trial"])]
                        for fold in trial_folds.to_dict("records"):
                            mlflow.log_metrics({
                                f"fold_{int(fold['fold'])}/game_weighted_RMSE": float(np.sqrt(fold["game_weighted_mse"])),
                                f"fold_{int(fold['fold'])}/game_weighted_MAE": float(fold["game_weighted_mae"]),
                            })
                        tuning_run_records.append({
                            "target_type": target_type, "trial": int(trial["trial"]),
                            "selected": bool(trial["selected"]),
                            "mlflow_trial_run_id": trial_run.info.run_id,
                            "mlflow_model_run_id": child.info.run_id,
                            "mlflow_parent_run_id": MLFLOW_PARENT_RUN_ID,
                            "mlflow_experiment_id": MLFLOW_EXPERIMENT_ID,
                        })
            mlflow.log_params({"model/" + k: str(v)
                               for k, v in pipeline.named_steps["model"].get_params().items()
                               if v is not None})
            mlflow.log_params({"forecast_months": FORECAST_MONTHS,
                               "clip_at_zero": info["clip_predictions_at_zero"],
                               "sales_month_monotonic": info["sales_month_monotonic"]})
            forecast = ForecastModel(pipeline, list(numeric_features), list(categorical_features),
                                     FORECAST_MONTHS, info["clip_predictions_at_zero"])
            rows = xgb_predictions_pdf.loc[xgb_predictions_pdf["target_type"].eq(target_type)]
            check = rows.loc[rows["split"].eq("test")]
            np.testing.assert_allclose(forecast.predict(check[feature_columns]), check["prediction"],
                                       rtol=1e-5, atol=1e-5)
            example = check[feature_columns].head(FORECAST_MONTHS).copy()
            model_name = {"name" if "name" in inspect.signature(mlflow.sklearn.log_model).parameters
                          else "artifact_path": "forecast"}
            saved = mlflow.sklearn.log_model(
                sk_model=forecast, input_example=example,
                signature=infer_signature(example, forecast.predict(example)),
                serialization_format="cloudpickle", pip_requirements=requirements, **model_name,
            )
            restored = mlflow.pyfunc.load_model(saved.model_uri)
            np.testing.assert_allclose(np.asarray(restored.predict(example)).reshape(-1),
                                       forecast.predict(example), rtol=1e-5, atol=1e-5)
            mlflow.set_tag("reload_prediction_check", "passed")
            mlflow_forecast_models[target_type] = saved.model_uri
            scores = xgb_monthly_metrics_pdf.loc[xgb_monthly_metrics_pdf["target_type"].eq(target_type)]
            overall = xgb_metrics_pdf.loc[xgb_metrics_pdf["target_type"].eq(target_type)]
            for estimator in ["xgboost", "monthly_mean"]:
                log_scores(overall.loc[overall["estimator"].eq(estimator)], f"test/{estimator}")
                log_scores(scores.loc[scores["estimator"].eq(estimator)], f"test/{estimator}")
            log_frame(xgb_split_summary_pdf.loc[xgb_split_summary_pdf["target_type"].eq(target_type)], "coverage")
            run_records.append({"target_type": target_type, "mlflow_model_run_id": child.info.run_id,
                                "model_uri": saved.model_uri})

    mlflow_runs_pdf = pd.DataFrame(run_records)
    mlflow_runs_pdf["mlflow_experiment_id"] = MLFLOW_EXPERIMENT_ID
    mlflow_runs_pdf["mlflow_parent_run_id"] = MLFLOW_PARENT_RUN_ID
    run_ids = dict(zip(mlflow_runs_pdf["target_type"], mlflow_runs_pdf["mlflow_model_run_id"]))
    for name, frame in {
        "predictions": xgb_predictions_pdf, "metrics": xgb_metrics_pdf,
        "monthly_metrics": xgb_monthly_metrics_pdf, "game_metrics": xgb_game_metrics_pdf,
        "split_summary": xgb_split_summary_pdf,
    }.items():
        frame["mlflow_experiment_id"] = MLFLOW_EXPERIMENT_ID
        frame["mlflow_parent_run_id"] = MLFLOW_PARENT_RUN_ID
        frame["mlflow_model_run_id"] = frame["target_type"].map(run_ids)
        log_frame(frame, name)
    tuning_mlflow_runs_pdf = pd.DataFrame(tuning_run_records)
    for name, frame in {
        "tuning_summary": tuning_summary_pdf, "tuning_trials": tuning_trials_pdf,
        "tuning_trial_folds": tuning_trial_folds_pdf, "tuning_folds": tuning_folds_pdf,
        "tuning_membership": tuning_membership_pdf, "tuning_runs": tuning_mlflow_runs_pdf,
    }.items():
        if not frame.empty:
            log_frame(frame, name)
    mlflow.log_dict(best_params_by_target, "selected_parameters.json")
    log_frame(mlflow_runs_pdf, "runs")
    log_frame(game_split_pdf.reset_index(), "game_split")
    if MLFLOW_LOG_DATA_SNAPSHOT:
        log_frame(model_pdf, "modeling_data")
    for (target_type, estimator), scores in xgb_monthly_metrics_pdf.groupby(["target_type", "estimator"]):
        log_scores(scores, f"{target_type}/{estimator}/test")
    for (target_type, estimator), scores in xgb_metrics_pdf.groupby(["target_type", "estimator"]):
        log_scores(scores, f"{target_type}/{estimator}/test")
    mlflow.log_dict(mlflow_forecast_models, "forecast_model_uris.json")
    mlflow.log_text("import mlflow\n\n# Inputs: cleaned numeric/categorical attributes + month_index.\n"
                    + "\n".join(f"{target}_model = mlflow.pyfunc.load_model({uri!r})\n"
                                f"{target}_predictions = {target}_model.predict(clean_inputs)\n"
                                for target, uri in mlflow_forecast_models.items()), "reload_forecasts.py")

print(f"MLflow experiment: {experiment_path}")
print(f"Parent run: {MLFLOW_PARENT_RUN_ID}")
print("Forecast models:", mlflow_forecast_models)
display(mlflow_runs_pdf)


In [0]:
# CELL 7 — Test accuracy by game, actual-demand bracket, and overall
# Run after cell 4; run cell 6 first to attach these results to the MLflow runs.
# Sales = cumulative units. Rentals = monthly units. No Eilers models are scored.
import math
import time
import tempfile
from pathlib import Path
import numpy as np
import pandas as pd
from pyspark.sql import functions as F, types as T

METRICS_TABLE = globals().get(
    "erro_table_path",
    "development_042_silver_sandbox.demand_forecast.forecast_error_metrics",
)
MIN_EVAL_MONTHS = 1
SAVE_METRICS = True
LOG_METRICS_TO_MLFLOW = True

IDENTITY = ["forecast_type", "metric_name", "revenue_type", "experiment_id", "experiment_name", "estimator"]
RUN_COLUMNS = ["mlflow_experiment_id", "mlflow_parent_run_id", "mlflow_model_run_id"]
GAME_KEYS = IDENTITY + ["game_key"]


def build_bracket_metrics(predictions, min_months=1):
    """Observed test pairs only; pooled row-level aggregates match the original evaluation convention."""
    if not isinstance(min_months, int) or min_months < 1:
        raise ValueError("MIN_EVAL_MONTHS must be a positive integer.")
    required = {"game_key", "target_type", "experiment_id", "experiment_name", "month_index",
                "prediction", "target_value", "split", "used_for_test", "is_observed", "target_is_imputed"}
    missing = required - set(predictions.columns)
    if missing:
        raise ValueError(f"Missing XGBoost prediction columns: {sorted(missing)}. Run cell 4 first.")
    selected = (
        predictions["split"].eq("test") & predictions["used_for_test"].eq(True).fillna(False)
        & predictions["is_observed"].eq(True).fillna(False)
        & predictions["target_is_imputed"].eq(False).fillna(False)
        & predictions["target_type"].isin(["sales", "rental"])
    )
    frame = predictions.loc[selected].drop(
        columns=["actual_bracket", "bracket_actual", "bracket_month_index", "bracket_rank", "cohort_test_games"],
        errors="ignore",
    ).copy()
    frame["actual"] = pd.to_numeric(frame["target_value"], errors="coerce")
    frame["prediction"] = pd.to_numeric(frame["prediction"], errors="coerce")
    frame["month_index"] = pd.to_numeric(frame["month_index"], errors="coerce")
    frame = frame.loc[np.isfinite(frame["actual"]) & np.isfinite(frame["prediction"])].copy()
    if frame.empty:
        raise ValueError("No finite, observed test forecast/actual pairs; nothing was written.")
    if not (frame["month_index"].notna() & frame["month_index"].ge(1)
            & frame["month_index"].eq(frame["month_index"].round())).all():
        raise ValueError("Scored month indices must be positive integers.")
    frame["month_index"] = frame["month_index"].astype(int)
    for c in ["game_key", "experiment_id", "experiment_name"]:
        if frame[c].isna().any() or frame[c].astype(str).str.strip().eq("").any():
            raise ValueError(f"Missing identity value in {c}.")
        frame[c] = frame[c].astype(str)
    frame["forecast_type"] = "demand"
    frame["metric_name"] = frame["target_type"].map({"sales": "cumulative_sales", "rental": "monthly_lease"})
    frame["revenue_type"] = frame["target_type"].map({"sales": "sales", "rental": "lease"})
    frame["estimator"] = "xgboost"
    frame["target_game"] = frame["game_key"]
    # game_key is the actual identifier in this notebook; no synthetic _target_id
    # or ownership classification is inferred from the revenue type.
    for c in RUN_COLUMNS:
        if c not in frame:
            frame[c] = None
    for _, group in frame.groupby(IDENTITY, sort=False):
        for c in RUN_COLUMNS:
            if group[c].dropna().nunique() > 1:
                raise ValueError(f"Multiple {c} values within one model/experiment. Select one logged run.")
    if frame.duplicated(GAME_KEYS + ["month_index"]).any():
        raise ValueError("Duplicate game/month test rows found; metrics were not written.")
    size = frame.groupby(GAME_KEYS)["month_index"].transform("size")
    frame = frame.loc[size.ge(min_months)].copy()
    if frame.empty:
        raise ValueError("No test games meet MIN_EVAL_MONTHS; nothing was written.")
    frame = frame.sort_values(GAME_KEYS + ["month_index"]).reset_index(drop=True)
    frame["error"] = frame["prediction"] - frame["actual"]
    frame["absolute_error"] = frame["error"].abs()
    frame["squared_error"] = frame["error"] ** 2
    previous = frame.groupby(GAME_KEYS)[["month_index", "prediction", "actual"]].shift()
    adjacent = frame["month_index"].eq(previous["month_index"] + 1)
    frame["direction_correct"] = np.nan
    frame.loc[adjacent, "direction_correct"] = (
        np.sign(frame.loc[adjacent, "prediction"] - previous.loc[adjacent, "prediction"])
        == np.sign(frame.loc[adjacent, "actual"] - previous.loc[adjacent, "actual"])
    ).astype(float)

    # Rank separately for each revenue type / experiment using latest SCORED actual.
    latest = frame.groupby(GAME_KEYS, sort=False).tail(1)
    brackets = latest[GAME_KEYS + ["target_game", "actual", "month_index"]].rename(
        columns={"actual": "bracket_actual", "month_index": "bracket_month_index"})
    brackets = brackets.sort_values(
        IDENTITY + ["bracket_actual", "target_game", "game_key"],
        ascending=[True] * len(IDENTITY) + [False, True, True],
    ).reset_index(drop=True)
    brackets["bracket_rank"] = brackets.groupby(IDENTITY).cumcount() + 1
    brackets["cohort_test_games"] = brackets.groupby(IDENTITY)["game_key"].transform("size")
    tail_size = np.ceil(brackets["cohort_test_games"] * 0.20).astype(int)
    brackets["actual_bracket"] = np.select(
        [brackets["bracket_rank"].le(tail_size),
         brackets["bracket_rank"].gt(brackets["cohort_test_games"] - tail_size)],
        ["top_20", "bottom_20"], default="middle_60")
    # Exact ties use the normalized game name as a stable tie-breaker, as in the example.
    # Small cohorts use ceil(20%); one-game cohorts are top_20 only.
    bracket_fields = ["actual_bracket", "bracket_actual", "bracket_month_index", "bracket_rank", "cohort_test_games"]
    frame = frame.merge(brackets[GAME_KEYS + bracket_fields], on=GAME_KEYS,
                        how="left", validate="many_to_one")
    output = []
    for level, group_columns in [
        ("per_game", GAME_KEYS), ("overall", IDENTITY), ("bracket", IDENTITY + ["actual_bracket"]),
    ]:
        for key, group in frame.groupby(group_columns, sort=False):
            row = dict(zip(group_columns, key))
            directions = group["direction_correct"].dropna()
            row.update({
                "result_level": level,
                "bracket": "all_test_games" if level == "overall" else group["actual_bracket"].iloc[0],
                "game_key": group["game_key"].iloc[0] if level == "per_game" else None,
                "target_game": group["target_game"].iloc[0] if level == "per_game" else None,
                "test_games": int(group["game_key"].nunique()), "valid_months": int(len(group)),
                "directional_comparisons": int(len(directions)),
                "mae": float(group["absolute_error"].mean()),
                "rmse": float(np.sqrt(group["squared_error"].mean())),
                "bias": float(group["error"].mean()),
                "directional_accuracy": float(directions.mean()) if len(directions) else None,
                "first_scored_month": int(group["month_index"].min()),
                "last_scored_month": int(group["month_index"].max()),
                "bracket_basis": "latest_scored_actual", "evaluation_split": "test",
            })
            for c in ["bracket_actual", "bracket_month_index", "bracket_rank", "cohort_test_games"]:
                row[c] = group[c].iloc[0] if level == "per_game" else None
            for c in RUN_COLUMNS:
                values = group[c].dropna()
                row[c] = str(values.iloc[0]) if len(values) else None
            output.append(row)
    metrics = pd.DataFrame(output).drop(columns="actual_bracket", errors="ignore")
    return frame, brackets, metrics


evaluation_rows_pdf, game_brackets_pdf, forecast_error_metrics_pdf = build_bracket_metrics(
    xgb_predictions_pdf, MIN_EVAL_MONTHS)

# Spark schema is explicit so single-month games with null directional accuracy
# and missing MLflow IDs work even when a whole column contains only nulls.
string_cols = ["result_level", *IDENTITY, "bracket", "game_key", "target_game",
               "bracket_basis", "evaluation_split", *RUN_COLUMNS]
long_cols = ["test_games", "valid_months", "directional_comparisons", "first_scored_month", "last_scored_month",
             "bracket_month_index", "bracket_rank", "cohort_test_games"]
double_cols = ["mae", "rmse", "bias", "directional_accuracy", "bracket_actual"]
output_columns = string_cols + long_cols + double_cols
schema = T.StructType(
    [T.StructField(c, T.StringType(), True) for c in string_cols]
    + [T.StructField(c, T.LongType(), True) for c in long_cols]
    + [T.StructField(c, T.DoubleType(), True) for c in double_cols])
records = []
for row in forecast_error_metrics_pdf[output_columns].to_dict("records"):
    records.append(tuple(None if pd.isna(row[c]) else
                         (str(row[c]) if c in string_cols else int(row[c]) if c in long_cols else float(row[c]))
                         for c in output_columns))
forecast_error_metrics_df = spark.createDataFrame(records, schema).withColumn("created_at", F.current_timestamp())
per_game_metrics_df = forecast_error_metrics_df.where(F.col("result_level") == "per_game")
overall_metrics_df = forecast_error_metrics_df.where(F.col("result_level") == "overall")
bracket_metrics_df = forecast_error_metrics_df.where(F.col("result_level") == "bracket")

# Attach the actual-demand bracket to all forecast months of each evaluated game.
# Training games and games without eligible test actuals retain null brackets.
join_keys = ["experiment_id", "experiment_name", "target_type", "game_key"]
game_flags = game_brackets_pdf.assign(
    target_type=game_brackets_pdf["revenue_type"].map({"sales": "sales", "lease": "rental"}))
flag_columns = ["actual_bracket", "bracket_actual", "bracket_month_index", "bracket_rank", "cohort_test_games"]
xgb_predictions_pdf = xgb_predictions_pdf.drop(columns=flag_columns, errors="ignore").merge(
    game_flags[join_keys + flag_columns], on=join_keys, how="left", validate="many_to_one")

if SAVE_METRICS:
    # Rerunning replaces this experiment's metrics and retains other experiments.
    # Schema evolution adds game_key / bracket metadata to an older shared table.
    experiments = sorted(forecast_error_metrics_pdf["experiment_id"].unique())
    predicate = "experiment_id IN (" + ",".join("'" + e.replace("'", "''") + "'" for e in experiments) + ")"
    writer = forecast_error_metrics_df.write.format("delta")
    if spark.catalog.tableExists(METRICS_TABLE):
        (writer.mode("overwrite").option("replaceWhere", predicate)
         .option("mergeSchema", "true").saveAsTable(METRICS_TABLE))
    else:
        writer.mode("errorifexists").saveAsTable(METRICS_TABLE)
    print(f"Saved metrics to {METRICS_TABLE}; replaced experiment(s): {experiments}")

if LOG_METRICS_TO_MLFLOW:
    # Explicit run IDs attach results to cell 6's completed runs without starting new ones.
    from mlflow.tracking import MlflowClient
    from mlflow.entities import Metric
    client = MlflowClient()
    linked = forecast_error_metrics_pdf.dropna(subset=["mlflow_model_run_id", "mlflow_parent_run_id"])
    if linked.empty:
        print("MLflow IDs are unavailable. Run cell 6, then rerun cell 7 to attach bracket metrics.")
    else:
        timestamp = int(time.time() * 1000)
        summary = linked.loc[linked["result_level"].isin(["overall", "bracket"])]
        batches = {}
        for row in summary.to_dict("records"):
            suffix = f"test_by_actual/{row['result_level']}/{row['bracket']}"
            for run_id, prefix in [(row["mlflow_model_run_id"], suffix),
                                   (row["mlflow_parent_run_id"], row["revenue_type"] + "/" + suffix)]:
                bucket = batches.setdefault(run_id, [])
                for metric in ["mae", "rmse", "bias", "directional_accuracy", "valid_months", "directional_comparisons", "test_games"]:
                    value = row[metric]
                    if pd.notna(value) and np.isfinite(value):
                        bucket.append(Metric(f"{prefix}/{metric}", float(value), timestamp, 0))
        for run_id, batch in batches.items():
            client.log_batch(run_id, metrics=batch)
        with tempfile.TemporaryDirectory(prefix="xgb_brackets_") as directory:
            for run_id, rows in linked.groupby("mlflow_parent_run_id"):
                path = Path(directory) / "test_bracket_and_game_metrics.csv"
                rows[output_columns].to_csv(path, index=False)
                client.log_artifact(run_id, str(path), artifact_path="evaluation")
                path = Path(directory) / "game_brackets.csv"
                experiments = set(rows["experiment_id"])
                game_brackets_pdf.loc[game_brackets_pdf["experiment_id"].isin(experiments)].to_csv(path, index=False)
                client.log_artifact(run_id, str(path), artifact_path="evaluation")
        print("Logged overall/bracket metrics and per-game/bracket CSVs to the existing MLflow runs.")

print("Brackets: latest scored actual per game and revenue type; ties use game name. Small cohorts use ceil(20%).")
print("Directional accuracy: 0–1, consecutive observed months only; flat matches flat; null if no comparisons.")
print("For cumulative sales, matching an upward direction does not imply accurate monthly sales increments.")
print("Overall/bracket errors pool game-month pairs, so longer observed histories contribute more rows.")
display(forecast_error_metrics_df.orderBy("experiment_id", "revenue_type", "result_level", "bracket", "target_game"))
